In [5]:
import json
from shapely.geometry import shape, Point

FIND_JSON_PATH = "find.json"
MACEIO_GEOJSON_PATH = "maceio_limite.geojson"
OUTPUT_IDS_JSON = "ids_agentes_maceio.json"
OUTPUT_IDS_TXT = "ids_agentes_maceio.txt"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_polygon(geojson_path):
    gj = load_json(geojson_path)

    if gj.get("type") == "FeatureCollection":
        geom = gj["features"][0]["geometry"]
    elif gj.get("type") == "Feature":
        geom = gj["geometry"]
    else:
        geom = gj

    return shape(geom)

def normalize_find_json_structure(data):
    if isinstance(data, dict):
        if "rows" in data:
            return data["rows"]
        if "data" in data and isinstance(data["data"], dict) and "rows" in data["data"]:
            return data["data"]["rows"]
    return data

def point_is_in_maceio(lat, lon, polygon):
    if lat is None or lon is None:
        return False
    try:
        point = Point(float(lon), float(lat))
        return polygon.contains(point) or polygon.touches(point)
    except Exception:
        return False

def main():
    polygon = load_polygon(MACEIO_GEOJSON_PATH)
    data = load_json(FIND_JSON_PATH)
    data = normalize_find_json_structure(data)

    ids = []

    for item in data:
        loc = item.get("location") or {}
        lat = loc.get("latitude")
        lon = loc.get("longitude")
        agent_id = item.get("id")

        if agent_id and point_is_in_maceio(lat, lon, polygon):
            ids.append(agent_id)

    ids = sorted(set(ids))

    with open(OUTPUT_IDS_JSON, "w", encoding="utf-8") as f:
        json.dump(ids, f, ensure_ascii=False, indent=2)

    with open(OUTPUT_IDS_TXT, "w", encoding="utf-8") as f:
        f.write(str(ids))

    print(f"Total de IDs em Maceió: {len(ids)}")
    print(f"Salvo em {OUTPUT_IDS_JSON}")
    print(f"Salvo em {OUTPUT_IDS_TXT}")
    print(ids[:20])

if __name__ == "__main__":
    main()

Total de IDs em Maceió: 436
Salvo em ids_agentes_maceio.json
Salvo em ids_agentes_maceio.txt
[662, 693, 1502, 1903, 9255, 10063, 13949, 15992, 17318, 18070, 18106, 18108, 18114, 18115, 18116, 18145, 18146, 18147, 18152, 18153]


In [7]:
url = "https://mapa.cultura.gov.br/api/agent/findOne?id=EQ%282754767%29&%40select=*"
# requisicao url
import requests
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:    
    print(f"Erro na requisição: {response.status_code}")

Erro na requisição: 403


In [8]:
import json
import pandas as pd
from pathlib import Path


ARQUIVO_ENTRADA = "agentes.json"
ARQUIVO_SAIDA_CSV = "agentes_tabela_principal.csv"
ARQUIVO_SAIDA_JSON = "agentes_tabela_principal.json"


def to_str_or_none(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value if value else None
    return str(value)


def get_nested(data, path, default=None):
    """
    Busca segura em dicionários aninhados.
    Ex: get_nested(obj, ["files", "avatar", "url"])
    """
    current = data
    for key in path:
        if not isinstance(current, dict):
            return default
        current = current.get(key)
        if current is None:
            return default
    return current


def first_non_empty(*values):
    for v in values:
        if v is None:
            continue
        if isinstance(v, str) and v.strip():
            return v.strip()
        elif not isinstance(v, str):
            return v
    return None


def extract_first_gallery_image(files_data):
    """
    Tenta encontrar a primeira imagem de galeria fora do avatar.
    A API pode variar, então o código olha vários formatos possíveis.
    """
    if not isinstance(files_data, dict):
        return None

    # grupos comuns possíveis
    possible_groups = [
        "gallery",
        "galeria",
        "image",
        "images",
        "img",
        "imagens",
    ]

    for group_name in possible_groups:
        group = files_data.get(group_name)

        if isinstance(group, list) and group:
            first_item = group[0]
            if isinstance(first_item, dict):
                return first_non_empty(
                    first_item.get("url"),
                    get_nested(first_item, ["transformations", "avatarBig", "url"]),
                    get_nested(first_item, ["transformations", "avatarMedium", "url"]),
                    get_nested(first_item, ["transformations", "avatarSmall", "url"]),
                )

        if isinstance(group, dict):
            # caso venha um objeto em vez de lista
            return first_non_empty(
                group.get("url"),
                get_nested(group, ["transformations", "avatarBig", "url"]),
                get_nested(group, ["transformations", "avatarMedium", "url"]),
                get_nested(group, ["transformations", "avatarSmall", "url"]),
            )

    # fallback: procurar qualquer arquivo que não seja avatar
    for key, value in files_data.items():
        if key == "avatar":
            continue

        if isinstance(value, list) and value:
            first_item = value[0]
            if isinstance(first_item, dict):
                url = first_non_empty(
                    first_item.get("url"),
                    get_nested(first_item, ["transformations", "avatarBig", "url"]),
                    get_nested(first_item, ["transformations", "avatarMedium", "url"]),
                    get_nested(first_item, ["transformations", "avatarSmall", "url"]),
                )
                if url:
                    return url

        elif isinstance(value, dict):
            url = first_non_empty(
                value.get("url"),
                get_nested(value, ["transformations", "avatarBig", "url"]),
                get_nested(value, ["transformations", "avatarMedium", "url"]),
                get_nested(value, ["transformations", "avatarSmall", "url"]),
            )
            if url:
                return url

    return None


def extract_phone(agent):
    """
    Tenta encontrar telefone em vários campos possíveis.
    """
    possible_paths = [
        ["telefonePublico"],
        ["telefone1"],
        ["telefone2"],
        ["telefone"],
        ["phone"],
        ["contatoPublico"],
        ["whatsapp"],
    ]

    for path in possible_paths:
        value = get_nested(agent, path)
        if isinstance(value, list):
            value = " | ".join([str(v).strip() for v in value if str(v).strip()])
        value = to_str_or_none(value)
        if value:
            return value

    return None


def extract_email(agent):
    """
    Tenta encontrar email em vários campos possíveis.
    """
    possible_paths = [
        ["emailPublico"],
        ["emailPrivado"],
        ["email"],
        ["publicEmail"],
    ]

    for path in possible_paths:
        value = to_str_or_none(get_nested(agent, path))
        if value:
            return value

    return None


def extract_area(agent):
    """
    Junta áreas/categorias principais.
    No exemplo, 'terms.area' existe e é uma lista. :contentReference[oaicite:2]{index=2}
    """
    areas = get_nested(agent, ["terms", "area"], [])
    if isinstance(areas, list):
        areas = [str(a).strip() for a in areas if str(a).strip()]
        return " | ".join(areas) if areas else None

    if isinstance(areas, str) and areas.strip():
        return areas.strip()

    # fallback possível
    category = get_nested(agent, ["rcv_registration", "category"])
    return to_str_or_none(category)


def extract_avatar(agent):
    """
    Pega a primeira foto/avatar.
    No exemplo, files.avatar.url existe. :contentReference[oaicite:3]{index=3}
    """
    return first_non_empty(
        get_nested(agent, ["files", "avatar", "url"]),
        get_nested(agent, ["avatar", "avatarBig", "url"]),
        get_nested(agent, ["avatar", "avatarMedium", "url"]),
        get_nested(agent, ["avatar", "avatarSmall", "url"]),
    )


def normalize_agent_record(item):
    """
    Normaliza cada item do JSON.
    Alguns registros vêm como {"id": ..., "data": {...}}.
    """
    agent = item.get("data", item)

    nome = to_str_or_none(agent.get("name"))
    bairro = to_str_or_none(agent.get("En_Bairro"))
    cidade = first_non_empty(
        to_str_or_none(agent.get("En_Municipio")),
        to_str_or_none(agent.get("geoMunicipio")),
    )

    lat = to_str_or_none(get_nested(agent, ["location", "latitude"]))
    lon = to_str_or_none(get_nested(agent, ["location", "longitude"]))

    long_description = first_non_empty(
        to_str_or_none(agent.get("longDescription")),
        to_str_or_none(agent.get("shortDescription")),
    )

    telefone = extract_phone(agent)
    email = extract_email(agent)
    area_categoria = extract_area(agent)
    foto_avatar = extract_avatar(agent)
    foto_galeria = extract_first_gallery_image(agent.get("files", {}))

    return {
        "id": agent.get("id"),
        "nome": nome,
        "bairro": bairro,
        "cidade": cidade,
        "lat": lat,
        "long": lon,
        "longDescription": long_description,
        "telefone": telefone,
        "email": email,
        "area_categoria": area_categoria,
        "foto_avatar": foto_avatar,
        "foto_galeria": foto_galeria,
    }


def main():
    caminho = Path(ARQUIVO_ENTRADA)

    with caminho.open("r", encoding="utf-8") as f:
        dados = json.load(f)

    if not isinstance(dados, list):
        raise ValueError("O arquivo agentes.json deve ser uma lista de registros.")

    registros = [normalize_agent_record(item) for item in dados]
    df = pd.DataFrame(registros)

    # opcional: remover duplicados por id
    if "id" in df.columns:
        df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    print(df.head())

    df.to_csv(ARQUIVO_SAIDA_CSV, index=False, encoding="utf-8-sig")
    df.to_json(ARQUIVO_SAIDA_JSON, orient="records", force_ascii=False, indent=2)

    print(f"CSV salvo em: {ARQUIVO_SAIDA_CSV}")
    print(f"JSON salvo em: {ARQUIVO_SAIDA_JSON}")
    print(f"Total de registros: {len(df)}")


if __name__ == "__main__":
    main()

     id                                               nome          bairro  \
0   662  Associacao De Homo, Hetero E Bisexuais (traves...          Centro   
1   693                   Orquestra de Tambores de Alagoas  Vergel do Lago   
2  1502                                   Coletivo Popfuzz     Pitanguinha   
3  1903                           Posse Atitude Periférica          Levada   
4  9255                                    PATRICIA SANTOS     Santa Lúcia   

   cidade                lat                 long  \
0  Maceió          -9.663537  -35.740521400000034   
1  Maceió  -9.66177217306344    -35.7362401485443   
2  Maceió         -9.6349266   -35.73249820000001   
3  Maceió         -9.6571251          -35.7465181   
4  Maceió         -9.5838001   -35.76369729999999   

                                     longDescription         telefone  \
0  Associacao De Homo, Hetero E Bisexuais (traves...             None   
1  A Orquestra de Tambores de Alagoas (OTA) surgi...  (82) 98867-0